In [1]:
import pandas as pd
import os
import re

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [11]:
def _fix_mojibake_text(x):
    if not isinstance(x, str):
        return x

    s = x
    if not any(ch in s for ch in ("Ã", "Â", "¤", "¦", "¥", "œ", "�")):
        return s

    candidates = [s]
    for enc in ("latin1", "cp1252"):
        try:
            candidates.append(s.encode(enc).decode("utf-8"))
        except Exception:
            pass

    def score(t):
        bad = sum(t.count(ch) for ch in ("Ã", "Â", "�"))
        return (bad, -sum(ord(ch) > 127 for ch in t))

    best = min(candidates, key=score)
    return best

p = 'AU/bio_data.csv'
df = pd.read_csv(p)
df_edited = df.copy()

for col in df_edited.select_dtypes(include=["object"]).columns:
    df_edited[col] = df_edited[col].map(_fix_mojibake_text)

out_p = os.path.join(os.path.dirname(p), "bio_data_edited.csv")
df_edited.to_csv(out_p, index=False, encoding="utf-8-sig")

print(f"Saved: {out_p}")

C:\Users\hamod4691\AppData\Local\Temp\ipykernel_17784\3691843218.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_edited.select_dtypes(include=["object"]).columns:


Saved: AU\bio_data_edited.csv


In [ ]:
p = 'AU/bio_data_edited.csv'
df = pd.read_csv(p)

qualifier_pattern = re.compile(
    r'\([^)]*\)|'                       # parentheses (keep them)
    r'(?i:\bPamiut\b)|'                 # specific place-name qualifier
    r'(?i:\bsp\.?\d+\b)|'             # sp2 or sp.2
    r'(?i:\.?juv\.?)(?=\W|$)|'         # .juv or juv.
    r'(?i:gen\. et spp\.?)|'            # gen. et spp.
    r'(?i:\b(?:spp|sp|indet|conf|var)\.?)(?=\W|$)|'  # spp/sp/indet/conf/var
    r'\bK\d(?:-\d){2}\b|'             # K-codes like K3-2-3
    r'\b[A-Z]{1,}[0-9]+(?:-[0-9]+)?\b|' # codes like NKF22-4, DW31-1, T-1 (letters+digits)
    r'\b[A-Z](?:-\d+|\d+)?\b|'        # lone capital letters maybe with -number (e.g. T-1)
    r'\b\d+\b|'                       # standalone numbers
    r'\?+'                              # question marks
)


def _extract_qualifiers(text):
    matches = list(qualifier_pattern.finditer(text))
    qualifiers = [match.group(0).strip() for match in matches]

    parts = []
    last_end = 0
    for match in matches:
        parts.append(text[last_end:match.start()])
        last_end = match.end()
    parts.append(text[last_end:])

    cleaned = re.sub(r'\s+', ' ', ''.join(parts)).strip(' ,;.-')
    if cleaned == '':
        cleaned = pd.NA

    return cleaned, qualifiers


def split_qualifier(s):
    if pd.isna(s):
        return pd.Series([pd.NA, pd.NA])

    orig = str(s)
    if '/' in orig:
        left, right = orig.split('/', 1)
        cleaned, qualifiers = _extract_qualifiers(left)
        right_cleaned, right_qualifiers = _extract_qualifiers(right)
        qualifiers.extend(right_qualifiers)
        if pd.notna(right_cleaned):
            qualifiers.append(str(right_cleaned).strip())
    else:
        cleaned, qualifiers = _extract_qualifiers(orig)

    qual = '; '.join(q for q in qualifiers if q and q.strip()) if qualifiers else pd.NA
    return pd.Series([cleaned, qual])


df[['scientificName', 'identificationQualifier']] = df['Taxon edited'].apply(split_qualifier)
# write CSV (may raise if file is locked)
df.to_csv(p, index=False)
# also write Excel
try:
    df.to_excel(os.path.splitext(p)[0] + '.xlsx', index=False)
except Exception as e:
    print('Excel write failed:', e)

In [15]:
# Remove rows where scientificName is "ovae" or "ukendt" (case/space insensitive)
to_remove = {"ovae", "ukendt"}
mask_remove = (
    df["scientificName"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(to_remove)
)

df = df.loc[~mask_remove].copy()

# Overwrite CSV and Excel with the same filenames
df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Removed {mask_remove.sum()} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Removed 3 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [19]:
old_name = "Ampharete finmarchia"
new_name = "Ampharete finmarchica"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 10 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [22]:
old_name = "Ampelisca eschrichti"
new_name = "Ampelisca eschrichtii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 32 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [23]:
old_name = "Ampharete goesi"
new_name = "Ampharete goësi"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 15 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [24]:
old_name = "Amphitite"
new_name = "Amphitrite"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [25]:
old_name = "Byblis gaimardi"
new_name = "Byblis gaimardii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [26]:
old_name = "Bylgides groenlandica"
new_name = "Bylgides groenlandicus"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [28]:
old_name = "Cingula moerchi"
new_name = "Alvania moerchii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 0 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [29]:
old_name = "Cingula moercki"
new_name = "Alvania moerchii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [30]:
old_name = "Cossura longicirrata"
new_name = "Cossura longocirrata"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 16 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [31]:
old_name = "Diastylis scorpipides"
new_name = "Diastylis scorpioides"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [32]:
old_name = "Microphthalmus sczelkowi"
new_name = "Microphthalmus sczelkowii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [ ]:
old_name = "Monoculodes kroyeri"
new_name = "Rostroculodes kroyeri"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [34]:
old_name = "Monoculodes krøyeri"
new_name = "Rostroculodes kroyeri"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 4 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [35]:
old_name = "Mya ånderør"
new_name = "Mya sp."

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 8 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [36]:
old_name = "Nainereis quadricuspida"
new_name = "Naineris quadricuspida"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [37]:
old_name = "Nebalia biceps"
new_name = "Nebalia bipes"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 2 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [38]:
old_name = "Nephtys longosetosus"
new_name = "Nephtys longosetosa"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [39]:
old_name = "Nicomache trispina"
new_name = "Nicomache trispinata"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [40]:
old_name = "Nudibranch"
new_name = "Nudibranchia"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 8 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [41]:
old_name = "Octocoral"
new_name = "Octocorallia"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [43]:
old_name = "Oediceros sagnisatus"
new_name = "Oediceros saginatus"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 27 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [44]:
old_name = "Oediceridae"
new_name = "Oedicerotidae"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 0 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [45]:
old_name = "Oenopota turricola"
new_name = "Propebela turricula"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 3 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [46]:
old_name = "Oenopota turricula"
new_name = "Propebela turricula"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 3 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [47]:
old_name = "Ophiocanta bidentata"
new_name = "Ophiacantha bidentata"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [48]:
old_name = "Ophiura sarsi"
new_name = "Ophiura sarsii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 10 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [49]:
old_name = "Opisa eschrichti"
new_name = "Opisa eschrichtii"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 1 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [50]:
old_name = "Pholoë"
new_name = "Pholoe"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 97 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [ ]:
old_name = "Pholoë"
new_name = "Pholoe"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

In [51]:
old_name = "Phoxocephalus holboelli"
new_name = "Phoxocephalus holbolli"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 41 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [53]:
old_name = "Phoxocephalus holbølli"
new_name = "Phoxocephalus holbolli"

mask = df["scientificName"].astype(str).str.strip().eq(old_name)
count = mask.sum()

df.loc[mask, "scientificName"] = new_name

df.to_csv(p, index=False, encoding="utf-8-sig")
df.to_excel(os.path.splitext(p)[0] + ".xlsx", index=False)

print(f"Updated {count} rows and saved:\n- {p}\n- {os.path.splitext(p)[0] + '.xlsx'}")

Updated 12 rows and saved:
- AU/bio_data_edited.csv
- AU/bio_data_edited.xlsx


In [21]:
unique_names = (
    df["scientificName"]
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda s: s.ne("")]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

unique_names_df = pd.DataFrame({"scientificName": unique_names})
unique_names_df.to_csv("AU/unique_names.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(unique_names_df)} unique names to unique_names.csv")
unique_names_df.head()

Saved 1178 unique names to unique_names.csv


,scientificName
0,Abyssoninoe
1,Abyssoninoe abyssorum
2,Acanthonotozoma magnum
3,Acanthonotozoma rusanovae
4,Acanthonotozoma serratum
